# EXPERIMENT 2 — MODERN SCHEMA MODEL (2016 - 2018)

## Objective

Test whether the 14 bureau tradeline fields (`open_acc_6m`, `il_util`, `all_util`, etc.) — which are only fully available from 2015 onward — add meaningful predictive value when trained on modern, fully-populated data. Experiment 1 always has these columns ~97–98% missing in training, so this experiment isolates their effect by training entirely on 2016–2018 loans, where the fields are complete.

In [1]:
import psutil
import pandas as pd
import numpy as np
import os
import gc
import time

mem = psutil.virtual_memory()

print(f"Total RAM: {mem.total / 1e9:.2f} GB")

print(f"Available: {mem.available / 1e9:.2f} GB")

print(f"Used: {mem.percent}%")

Total RAM: 8.22 GB
Available: 1.34 GB
Used: 83.7%


In [2]:
# ============================================================
#  Load raw data, filter to 2016-2018
# ============================================================
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

DATA_PATH = os.path.join(project_root, 'src', 'data', 'raw', 'accepted_2007_to_2018Q4.csv')

CHUNK_SIZE = 100_000

start = time.time()

filtered_chunks = []

total_rows_seen = 0

# chunk is a batch of 100,000 raw rows straight from the CSV 
for chunk in pd.read_csv(DATA_PATH, chunksize = CHUNK_SIZE, low_memory = False):
    
    total_rows_seen += len(chunk)
    
# Parse issue_d and filter to 2016-2018 only
    chunk['issue_d'] = pd.to_datetime(chunk['issue_d'], format = '%b-%Y', errors = 'coerce'
                                     )
    chunk_filtered = chunk[chunk['issue_d'].dt.year.isin([2016, 2017, 2018])]

    if len(chunk_filtered) > 0:
        
        filtered_chunks.append(chunk_filtered)

    del chunk    # free memory for the discarded rows immediately

print(f"Total rows scanned: {total_rows_seen:,}")

df = pd.concat(filtered_chunks, ignore_index = True)

del filtered_chunks

print(f"Rows after year filter (2016-2018): {len(df):,}")

print(f"Load time: {time.time() - start:.1f}s")

Total rows scanned: 2,260,701
Rows after year filter (2016-2018): 1,373,228
Load time: 78.2s


In [3]:
df.shape

(1373228, 151)

In [4]:
# ============================================================
# Drop leakage/identifier/redundant columns
# ============================================================
FINAL_DROP_LIST = [

    # === IDENTIFIERS ===
    'id',
    'member_id',
    'url',
    'policy_code',
     
    # === HIGH CARDINALITY / UNSTRUCTURED TEXT ===
    'title',         # borrower free text — high cardinality, noise
    'zip_code',      # High-cardinality geographic feature; excluded due to privacy/proxy-risk concerns
    
    # === REDUNDANT FEATURES — High correlation ===
    'funded_amnt',
    'funded_amnt_inv',
    'num_sats',
    'num_rev_tl_bal_gt_0',

    # === POST-LOAN PAYMENTS & RECOVERY — Leakage ===
    'out_prncp',
    'out_prncp_inv',
    'total_pymnt',
    'total_pymnt_inv',
    'total_rec_prncp',
    'total_rec_int',
    'total_rec_late_fee',
    'recoveries',
    'collection_recovery_fee',
    'last_pymnt_amnt',

    # === POST-LOAN DATES — Leakage ===
    'last_pymnt_d',
    'next_pymnt_d',
    'last_credit_pull_d',

    # === POST-LOAN FICO — Leakage ===
    'last_fico_range_high',
    'last_fico_range_low',

    # === SETTLEMENT — Leakage ===
    'debt_settlement_flag',
    'debt_settlement_flag_date',
    'settlement_status',
    'settlement_date',
    'settlement_amount',
    'settlement_percentage',
    'settlement_term',

    # === HARDSHIP — Leakage ===
    'hardship_flag',
    'hardship_type',
    'hardship_reason',
    'hardship_status',
    'hardship_start_date',
    'hardship_end_date',
    'payment_plan_start_date',
    'hardship_dpd',
    'hardship_loan_status',
    'hardship_payoff_balance_amount',
    'hardship_amount',
    'orig_projected_additional_accrued_interest',
    'hardship_last_payment_amount',
    'hardship_length',
    'deferral_term',

    # === PAYMENT PLAN — ambiguous / post-loan behavior ===
    'pymnt_plan',

    # === SECONDARY APPLICANT — High missingness,sparse optional fields, ===
    'annual_inc_joint',
    'dti_joint',
    'verification_status_joint',
    'revol_bal_joint',
    'sec_app_num_rev_accts',
    'sec_app_fico_range_low',
    'sec_app_fico_range_high',
    'sec_app_earliest_cr_line',
    'sec_app_inq_last_6mths',
    'sec_app_mort_acc',
    'sec_app_open_acc',
    'sec_app_revol_util',
    'sec_app_open_act_il',
    'sec_app_chargeoff_within_12_mths',
    'sec_app_collections_12_mths_ex_med',
    'sec_app_mths_since_last_major_derog',

    # === HIGH MISSING / UNSTRUCTURED ===
    'desc'
]

cols_before = df.shape[1]
df = df.drop(columns=[c for c in FINAL_DROP_LIST if c in df.columns])
cols_after = df.shape[1]

print(f"Columns before drop: {cols_before}")
print(f"Columns dropped: {cols_before - cols_after}")
print(f"Columns after drop: {cols_after}")

Columns before drop: 151
Columns dropped: 65
Columns after drop: 86


In [5]:
# ============================================================
#  Datetime conversion (with dtype verification)
# ============================================================
date_cols = ['issue_d', 'earliest_cr_line']

for col in date_cols:
    df[col] = pd.to_datetime(df[col], format='%b-%Y', errors='coerce')

# Verify these are genuinely datetime64, not object
for col in date_cols:
    print(f"{col}: {df[col].dtype}")

issue_d: datetime64[ns]
earliest_cr_line: datetime64[ns]


In [6]:
# ============================================================
# Clean emp_length
# ============================================================
def clean_emp_length(col):
    
    col = col.str.replace(r'< 1 year', '0', regex=False)
    
    col = col.str.replace(r'\+ years', '', regex = True)
    
    col = col.str.replace(r' years', '', regex = True)
    
    col = col.str.replace(r' year' ,'', regex = True)

    return pd.to_numeric(col, errors = 'coerce')

df['emp_length'] = clean_emp_length(df['emp_length'])
print(df['emp_length'].dtype)

float64


In [7]:
# =========================================================
# Dtype Optimization
# =========================================================

import pandas as pd

def optimize_dtypes(df):

    # -----------------------------------------------------
    # Optimize Float Columns | float64 -> float32
    # -----------------------------------------------------
    
    float_cols = df.select_dtypes(include=['float64']).columns

    for col in float_cols:
        
        df[col] = pd.to_numeric(df[col], downcast='float')

    # -----------------------------------------------------
    # Optimize Integer Columns | int64 -> int32/int16/int8
    # -----------------------------------------------------
    
    int_cols = df.select_dtypes(include=['int64']).columns

    for col in int_cols:
        
        df[col] = pd.to_numeric(df[col], downcast='integer')

    # -----------------------------------------------------
    # Convert High-Cardinality Text Columns to String
    # -----------------------------------------------------
    
    high_cardinality_cols = [
        'emp_title'
    ]

    for col in high_cardinality_cols:

        if col in df.columns:
            
            df[col] = df[col].astype('string')

    # -----------------------------------------------------
    # Convert Target Column to Category
    # -----------------------------------------------------
    
    if 'loan_status' in df.columns:
        
        df['loan_status'] = df['loan_status'].astype('category')

    # -----------------------------------------------------
    # Convert Low-Cardinality Columns to Category
    # -----------------------------------------------------
    
    category_cols = [
        'grade',
        'sub_grade',
        'term',
        'home_ownership',
        'verification_status',
        'purpose',
        'addr_state',
        'initial_list_status',
        'application_type',
        'disbursement_method'
        ]

    for col in category_cols:

        if col in df.columns:
            
            df[col] = df[col].astype('category')

    # -----------------------------------------------------
    # Return Optimized DataFrame
    # -----------------------------------------------------
    
    return df

df = optimize_dtypes(df)

In [8]:
print(df.info(memory_usage='deep'))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1373228 entries, 0 to 1373227
Data columns (total 86 columns):
 #   Column                          Non-Null Count    Dtype         
---  ------                          --------------    -----         
 0   loan_amnt                       1373228 non-null  float32       
 1   term                            1373228 non-null  category      
 2   int_rate                        1373228 non-null  float32       
 3   installment                     1373228 non-null  float32       
 4   grade                           1373228 non-null  category      
 5   sub_grade                       1373228 non-null  category      
 6   emp_title                       1257732 non-null  string        
 7   emp_length                      1271156 non-null  float32       
 8   home_ownership                  1373228 non-null  category      
 9   annual_inc                      1373228 non-null  float64       
 10  verification_status             1373228 no

In [9]:
# =========================================================
# Target variable
# =========================================================
DEFAULT_STATUSES = [
    'Charged Off', 'Default', 'Late (31-120 days)',
    'Does not meet the credit policy. Status:Charged Off',
]
NON_DEFAULT_STATUSES = [
    'Fully Paid', 'Does not meet the credit policy. Status:Fully Paid',
]
 
MATURITY_THRESHOLD_PCT = 1.0  
SNAPSHOT_DATE = pd.Timestamp('2018-12-31')
 
def recover_mature_current_loans(df, snapshot_date, maturity_threshold_pct):
    """A 'Current' loan that has already survived its full term (relative to
    its own term length) can be trusted as non-default, instead of being
    dropped just because the dataset snapshot cuts it off mid-story."""
    months_since_issue = (
        (snapshot_date.year - df['issue_d'].dt.year) * 12
        + (snapshot_date.month - df['issue_d'].dt.month)
    )
    term_months = df['term'].astype(str).str.extract(r'(\d+)').astype(float)
    term_months = term_months.fillna(term_months.median())
    pct_of_term_elapsed = months_since_issue / term_months.iloc[:, 0]
    return (df['loan_status'] == 'Current') & (pct_of_term_elapsed >= maturity_threshold_pct)
 
mask_mature_current = recover_mature_current_loans(df, SNAPSHOT_DATE, MATURITY_THRESHOLD_PCT)
print("Mature 'Current' loans recovered:", mask_mature_current.sum())
 
mask_keep = df['loan_status'].isin(DEFAULT_STATUSES + NON_DEFAULT_STATUSES) | mask_mature_current
df = df[mask_keep].copy()
del mask_keep, mask_mature_current
gc.collect()

Mature 'Current' loans recovered: 0


40

In [10]:
df['target'] = df['loan_status'].isin(DEFAULT_STATUSES).astype('int8')
print(df['target'].value_counts())
print(df['target'].value_counts(normalize=True) * 100)
 
df = df.drop(columns=['loan_status'])

target
0    402449
1    136066
Name: count, dtype: int64
target
0    74.733109
1    25.266891
Name: proportion, dtype: float64


In [11]:
target_pct = df['target'].value_counts(normalize = True) * 100

print(f" 0(Non- Default) : {target_pct[0] :.2f}% ")

print(f" 1( Default) : {target_pct[1] :.2f}% ")

 0(Non- Default) : 74.73% 
 1( Default) : 25.27% 


In [12]:
# =========================================================
#  Data quality fixes
# =========================================================
df.loc[df['annual_inc'] == 0, 'annual_inc'] = np.nan
 
bad_income_mask = (df['annual_inc'] < 5000) & (df['loan_amnt'] / df['annual_inc'] > 3)
df.loc[bad_income_mask, 'annual_inc'] = np.nan
print(f"Rows flagged by income/loan ratio rule: {bad_income_mask.sum()}")
del bad_income_mask
 
# dti == -1 sentinel 
df.loc[df['dti'] == -1, 'dti'] = np.nan
 
df.loc[df['tot_hi_cred_lim'] == 9_999_999, 'tot_hi_cred_lim'] = np.nan
df.loc[df['total_rev_hi_lim'] == 9_999_999, 'total_rev_hi_lim'] = np.nan

Rows flagged by income/loan ratio rule: 148


In [13]:
# Binary flags — created 
df['ever_delinquent']       = df['mths_since_last_delinq'].notna().astype('int8')
df['has_public_record']     = df['mths_since_last_record'].notna().astype('int8')
df['has_bc_delinquency']    = df['mths_since_recent_bc_dlq'].notna().astype('int8')
df['has_major_derog']       = df['mths_since_last_major_derog'].notna().astype('int8')
df['has_installment_accts'] = df['il_util'].notna().astype('int8')
df['has_revol_delinq']      = df['mths_since_recent_revol_delinq'].notna().astype('int8')
df['has_bankcard']          = df['bc_util'].notna().astype('int8')
df['has_rcnt_il']           = df['mths_since_rcnt_il'].notna().astype('int8')

In [14]:
# =========================================================
# 9. Sentinel fill for delinquency "months since" columns
# =========================================================
sentinel_cols = [
    'mths_since_last_delinq', 'mths_since_last_record', 'mths_since_recent_bc_dlq',
    'mths_since_last_major_derog', 'mths_since_recent_revol_delinq',
]
for col in sentinel_cols:
    df[col] = df[col].fillna(999).astype('float32')

In [15]:
# =========================================================
# FEATURE ENGINEERING
# =========================================================

def engineer_features(df):

    # --- FICO midpoint ---
    # Two separate columns in raw data -> one clean score
    df['fico_score'] = (
        (df['fico_range_low'] + df['fico_range_high']) / 2
    ).astype('float32')


    # --- Payment-to-income ratio ---
    # What fraction of monthly income goes to this loan repayment
    # Higher = more financial stress = higher default risk
    df['pti_ratio'] = (
        df['installment'] / (df['annual_inc'] / 12)
    ).astype('float32')


    # --- Loan-to-income ratio ---
    # How large is this loan relative to annual income
    # Higher = borrower taking on more than they can handle
    df['loan_to_income'] = (
        df['loan_amnt'] / df['annual_inc']
    ).astype('float32')


    # --- Overall credit utilization ---
    # +1 avoids division by zero for borrowers with no credit limit recorded
    df['overall_util'] = (
        df['tot_cur_bal'] / (df['tot_hi_cred_lim'] + 1)
    ).astype('float32')


    # --- Credit history age ---
    # Older history usually indicates more stable borrowers
    df['credit_age_yrs'] = (
        (df['issue_d'] - df['earliest_cr_line']).dt.days / 365
    ).astype('float32')


    # --- Loan issue year ---
    # Captures economic cycle effects
    df['issue_year'] = (
        df['issue_d'].dt.year
    ).astype('Int16')
    
    return df

df =  engineer_features(df)

In [16]:
# ============================================================
# Drop columns superseded by engineered features
# ============================================================
cols_to_drop_post_fe = [
    'fico_range_low', 'fico_range_high', 'earliest_cr_line', 'installment',
    'emp_title', 'disbursement_method', 'application_type', 'grade',
]
df = df.drop(columns=[c for c in cols_to_drop_post_fe if c in df.columns])
print(f"Shape after post-FE drop: {df.shape}")

Shape after post-FE drop: (538515, 92)


In [17]:
# ============================================================
# Split by year (2016 train / 2017 val / 2018 test)
# ============================================================
train_e2 = df[df['issue_year'] == 2016].copy()
val_e2   = df[df['issue_year'] == 2017].copy()
test_e2  = df[df['issue_year'] == 2018].copy()

X_train_e2 = train_e2.drop(columns=['target'])
y_train_e2 = train_e2['target']
X_val_e2   = val_e2.drop(columns=['target'])
y_val_e2   = val_e2['target']
X_test_e2  = test_e2.drop(columns=['target'])
y_test_e2  = test_e2['target']

print(f"Train (2016): {X_train_e2.shape} | default rate: {y_train_e2.mean():.2%}")
print(f"Val (2017):   {X_val_e2.shape} | default rate: {y_val_e2.mean():.2%}")
print(f"Test (2018):  {X_test_e2.shape} | default rate: {y_test_e2.mean():.2%}")

Train (2016): (297651, 91) | default rate: 24.46%
Val (2017):   (177325, 91) | default rate: 26.60%
Test (2018):  (63539, 91) | default rate: 25.33%


In [18]:
# ============================================================
#  Winsorization (99th percentile, fit on train only)
# ============================================================
winsorize_cols = [
    'annual_inc', 'tot_cur_bal', 'total_bal_ex_mort', 'revol_bal',
    'total_il_high_credit_limit', 'avg_cur_bal', 'total_bc_limit',
    'bc_open_to_buy', 'max_bal_bc', 'tot_coll_amt', 'delinq_amnt',
    'tot_hi_cred_lim', 'total_rev_hi_lim',
]

winsor_caps = {}
for col in winsorize_cols:
    if col in X_train_e2.columns:
        cap = X_train_e2[col].quantile(0.99)
        winsor_caps[col] = cap

for split_df in (X_train_e2, X_val_e2, X_test_e2):
    for col, cap in winsor_caps.items():
        split_df[col] = split_df[col].clip(upper=cap)

print("Winsorization caps (from train):")
for col, cap in winsor_caps.items():
    print(f"  {col}: {cap:.2f}")

Winsorization caps (from train):
  annual_inc: 272000.00
  tot_cur_bal: 687114.50
  total_bal_ex_mort: 242058.00
  revol_bal: 103928.50
  total_il_high_credit_limit: 209532.00
  avg_cur_bal: 73884.00
  total_bc_limit: 105200.00
  bc_open_to_buy: 74715.99
  max_bal_bc: 24434.30
  tot_coll_amt: 5735.00
  delinq_amnt: 0.00
  tot_hi_cred_lim: 784376.62
  total_rev_hi_lim: 163800.00


In [19]:
# =========================================================
# 1Sentinel + business rule caps for dti and utilization columns
# =========================================================
util_cols = ['revol_util', 'bc_util', 'il_util', 'all_util']
 
for split_df in (X_train_e2, X_val_e2, X_test_e2):
    # dti sentinel fix (separate from the earlier dti == -1 fix)
    split_df['dti'] = split_df['dti'].replace(999, np.nan)
 
    # utilization columns: business-rule cap (these ARE meant to be capped)
    for col in util_cols:
        if col in split_df.columns:
            split_df[col] = split_df[col].clip(upper=150)
 
    # dti: extreme values treated as missing, not capped
    split_df['dti'] = split_df['dti'].mask(split_df['dti'] > 65, np.nan)
 

In [20]:
# ============================================================
# log1p transform (applied after winsorizing/capping)
# ============================================================
log1p_cols = [
    'tot_coll_amt', 'bc_open_to_buy', 'total_rev_hi_lim', 'total_bal_ex_mort',
    'avg_cur_bal', 'annual_inc', 'tot_cur_bal', 'tot_hi_cred_lim',
]

for split_df in (X_train_e2, X_val_e2, X_test_e2):
    for col in log1p_cols:
        if col in split_df.columns:
            split_df[col] = np.log1p(split_df[col].clip(lower=0))  # clip(lower=0) guards against negative values before log1p

print("log1p applied to:", log1p_cols)

log1p applied to: ['tot_coll_amt', 'bc_open_to_buy', 'total_rev_hi_lim', 'total_bal_ex_mort', 'avg_cur_bal', 'annual_inc', 'tot_cur_bal', 'tot_hi_cred_lim']


In [21]:
# ============================================================
#  Imputation (only sentinel cols excluded)
# ============================================================
from sklearn.impute import SimpleImputer

numeric_cols_all = X_train_e2.select_dtypes(include=['float32', 'float64', 'int8', 'int16', 'int32', 'Int16']).columns.tolist()
numeric_cols = [c for c in numeric_cols_all if c not in sentinel_cols]

num_imputer = SimpleImputer(strategy='median')
X_train_e2[numeric_cols] = num_imputer.fit_transform(X_train_e2[numeric_cols])
X_val_e2[numeric_cols]   = num_imputer.transform(X_val_e2[numeric_cols])
X_test_e2[numeric_cols]  = num_imputer.transform(X_test_e2[numeric_cols])

categorical_cols = X_train_e2.select_dtypes(include=['category', 'object', 'string']).columns.tolist()
categorical_cols_for_impute = [c for c in categorical_cols ]

cat_imputer = SimpleImputer(strategy='most_frequent')
if categorical_cols_for_impute:
    X_train_e2[categorical_cols_for_impute] = cat_imputer.fit_transform(X_train_e2[categorical_cols_for_impute])
    X_val_e2[categorical_cols_for_impute]   = cat_imputer.transform(X_val_e2[categorical_cols_for_impute])
    X_test_e2[categorical_cols_for_impute]  = cat_imputer.transform(X_test_e2[categorical_cols_for_impute])

print(f"Numeric NaNs remaining (train): {X_train_e2[numeric_cols].isna().sum().sum()}")

Numeric NaNs remaining (train): 0


In [22]:
# =========================================================
# 18. Encoding
# =========================================================
initial_list_map = {'f': 0, 'w': 1}
for split_df in (X_train_e2, X_val_e2, X_test_e2):
    split_df['initial_list_status'] = split_df['initial_list_status'].map(initial_list_map).astype('int8')

grade_letters = ['A', 'B', 'C', 'D', 'E', 'F', 'G']
sub_grade_map = {f"{letter}{num}": (i * 5 + num)
                 for i, letter in enumerate(grade_letters) for num in range(1, 6)}
for split_df in (X_train_e2, X_val_e2, X_test_e2):
    split_df['sub_grade'] = split_df['sub_grade'].astype(str).map(sub_grade_map).astype('float32')

for split_df in (X_train_e2, X_val_e2, X_test_e2):
    split_df['term'] = split_df['term'].astype(str).str.extract(r'(\d+)').astype('float32')

verification_map = {'Not Verified': 0, 'Source Verified': 1, 'Verified': 2}
for split_df in (X_train_e2, X_val_e2, X_test_e2):
    split_df['verification_status'] = split_df['verification_status'].astype(str).map(verification_map).astype('int8')

def one_hot_encode(train_df, val_df, test_df, col, prefix=None, merge_map=None):
    prefix = prefix or col
    if merge_map:
        train_df, val_df, test_df = train_df.copy(), val_df.copy(), test_df.copy()
        train_df[col] = train_df[col].replace(merge_map)
        val_df[col] = val_df[col].replace(merge_map)
        test_df[col] = test_df[col].replace(merge_map)
    train_dummies = pd.get_dummies(train_df[col], prefix=prefix)
    dummy_cols = train_dummies.columns.tolist()[1:]

    def apply_dummies(split_df):
        dummies = pd.get_dummies(split_df[col], prefix=prefix)
        dummies = dummies.reindex(columns=dummy_cols, fill_value=0)
        return pd.concat([split_df.drop(columns=[col]), dummies], axis=1)

    return apply_dummies(train_df), apply_dummies(val_df), apply_dummies(test_df)

X_train_e2, X_val_e2, X_test_e2 = one_hot_encode(
    X_train_e2, X_val_e2, X_test_e2, 'home_ownership', merge_map={'ANY': 'OTHER', 'NONE': 'OTHER'}
)

X_train_e2, X_val_e2, X_test_e2 = one_hot_encode(X_train_e2, X_val_e2, X_test_e2, 'purpose')

state_counts = X_train_e2['addr_state'].value_counts()
rare_states = state_counts[state_counts < 500].index.tolist()
merge_map = {s: 'OTHER_STATE' for s in rare_states}

X_train_e2, X_val_e2, X_test_e2 = one_hot_encode(
    X_train_e2, X_val_e2, X_test_e2, 'addr_state', merge_map=merge_map
)

In [23]:
 #=========================================================
# 19. Final drops
# =========================================================
tree_drop_flags = [
    'ever_delinquent',
    'has_public_record',
    'has_bc_delinquency',
    'has_major_derog',
    'has_revol_delinq',
]

cols_to_drop = [c for c in ['issue_year', 'issue_d', 'delinq_amnt'] if c in X_train_e2.columns]
cols_to_drop_final = cols_to_drop + tree_drop_flags

X_train_e2 = X_train_e2.drop(columns=cols_to_drop_final)
X_val_e2   = X_val_e2.drop(columns=cols_to_drop_final)
X_test_e2  = X_test_e2.drop(columns=cols_to_drop_final)
 
print(f"Final X_train_e2 shape: {X_train_e2.shape}")
print(f"Object/category dtypes remaining: {X_train_e2.select_dtypes(include=['object','category']).columns.tolist()}")
print(f"Total NaNs: {X_train_e2.isna().sum().sum()}")
print(f"X_val_e2  shape: {X_val_e2.shape}  | NaNs: {X_val_e2.isna().sum().sum()}")
print(f"X_test_e2 shape: {X_test_e2.shape} | NaNs: {X_test_e2.isna().sum().sum()}")

Final X_train_e2 shape: (297651, 144)
Object/category dtypes remaining: []
Total NaNs: 0
X_val_e2  shape: (177325, 144)  | NaNs: 0
X_test_e2 shape: (63539, 144) | NaNs: 0


In [24]:
# =========================================================
# 20. Evaluation helpers
# =========================================================
def evaluate_credit_model(y_true, y_proba, model_name="model", n_deciles=10, verbose=True):
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)
 
    roc_auc = roc_auc_score(y_true, y_proba)
    pr_auc = average_precision_score(y_true, y_proba)
    gini = 2 * roc_auc - 1
    brier = brier_score_loss(y_true, y_proba)
    ks_stat = ks_2samp(y_proba[y_true == 1], y_proba[y_true == 0]).statistic
 
    tmp = pd.DataFrame({'y_true': y_true, 'y_proba': y_proba})
    tmp['decile'] = pd.qcut(
        tmp['y_proba'].rank(method='first'), n_deciles,
        labels=[f"D{i}" for i in range(1, n_deciles + 1)]
    )
    lift_table = tmp.groupby('decile', observed=True).agg(
        n=('y_true', 'size'), default_rate=('y_true', 'mean')
    ).reindex([f"D{i}" for i in range(1, n_deciles + 1)])
    overall_rate = y_true.mean()
    lift_table['lift'] = lift_table['default_rate'] / overall_rate
    top10_default_rate = lift_table.loc['D10', 'default_rate']
    top10_lift = lift_table.loc['D10', 'lift']
 
    if verbose:
        print(f"=== {model_name} ===")
        print(f"ROC-AUC:      {roc_auc:.4f}")
        print(f"PR-AUC:       {pr_auc:.4f}")
        print(f"Gini:         {gini:.4f}")
        print(f"KS statistic: {ks_stat:.4f}")
        print(f"Brier score:  {brier:.4f}")
        print(f"Top 10% (D10) default rate: {top10_default_rate:.2%}  (lift: {top10_lift:.2f}x)")
 
    return {
        'model': model_name, 'roc_auc': roc_auc, 'pr_auc': pr_auc, 'gini': gini,
        'ks_stat': ks_stat, 'brier': brier,
        'top10_default_rate': top10_default_rate, 'top10_lift': top10_lift,
        'lift_table': lift_table,
    }
 
def evaluate_train_val(model, X_train, y_train, X_val, y_val, model_name='model'):
    train_proba = model.predict_proba(X_train)[:, 1]
    val_proba = model.predict_proba(X_val)[:, 1]
    train_results = evaluate_credit_model(y_train, train_proba, f"{model_name} - Train")
    print()
    val_results = evaluate_credit_model(y_val, val_proba, f"{model_name} - Val")
    gap = train_results['roc_auc'] - val_results['roc_auc']
    print(f"\nTrain/Val ROC-AUC gap: {gap:.4f}  "
          f"({'Good fit' if gap < 0.03 else 'Check for overfitting'})")
    return train_results, val_results
 

In [25]:
# =========================================================
# 21. Train XGBoost — Experiment 2
# =========================================================
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from scipy.stats import ks_2samp

scale_pos_weight_e2 = (y_train_e2 == 0).sum() / (y_train_e2 == 1).sum()
print(f"scale_pos_weight for 2016 train: {scale_pos_weight_e2:.2f}")
 
xgb_e2 = XGBClassifier(
    n_estimators=1500,
    max_depth=4,
    learning_rate=0.05,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight_e2,
    tree_method='hist',
    eval_metric='auc',
    early_stopping_rounds=30,
    n_jobs=2,
    random_state=42,
)
xgb_e2.fit(X_train_e2, y_train_e2, eval_set=[(X_val_e2, y_val_e2)], verbose=100)
print(f"Best iteration: {xgb_e2.best_iteration}")
 
train_results_e2, val_results_e2 = evaluate_train_val(
    xgb_e2, X_train_e2, y_train_e2, X_val_e2, y_val_e2, model_name="XGBoost — Experiment 2 (corrected)"
)

scale_pos_weight for 2016 train: 3.09
[0]	validation_0-auc:0.67795
[100]	validation_0-auc:0.70750
[200]	validation_0-auc:0.71453
[300]	validation_0-auc:0.71788
[400]	validation_0-auc:0.72009
[500]	validation_0-auc:0.72171
[600]	validation_0-auc:0.72287
[700]	validation_0-auc:0.72380
[800]	validation_0-auc:0.72442
[900]	validation_0-auc:0.72479
[1000]	validation_0-auc:0.72518
[1047]	validation_0-auc:0.72524
Best iteration: 1018
=== XGBoost — Experiment 2 (corrected) - Train ===
ROC-AUC:      0.7612
PR-AUC:       0.5064
Gini:         0.5224
KS statistic: 0.3819
Brier score:  0.2007
Top 10% (D10) default rate: 60.67%  (lift: 2.48x)

=== XGBoost — Experiment 2 (corrected) - Val ===
ROC-AUC:      0.7253
PR-AUC:       0.4786
Gini:         0.4505
KS statistic: 0.3261
Brier score:  0.2093
Top 10% (D10) default rate: 57.22%  (lift: 2.15x)

Train/Val ROC-AUC gap: 0.0359  (Check for overfitting)


### Final Summary

| Metric | Experiment 1 XGBoost (no tradeline, 2007–2015 train) | Experiment 2 XGBoost (with tradeline, 2016 train) | Difference |
|---|---|---|---|
| Val ROC-AUC | 0.7254 | 0.7253 | -0.0001 |
| Val KS | 0.3267 | 0.3261 | -0.0006 |
| Val Brier | 0.2145 | 0.2093 | -0.0052 |
| Train/Val Gap | 0.0322 (good) | 0.0359 (mild overfit) | +0.0037 |
| Training rows | 623,267 | 297,651 | -52% |

Both models scored almost the same on validation. Experiment 2 used half the training data but still matched Experiment 1, with a slightly wider overfitting gap.

This result decided which candidates moved on to test-set evaluation — not the final model. Both were tested on 2017–2018 data afterward.

**What happened on test:** Experiment 2 actually scored higher on every metric there too. But Experiment 1 was still chosen as the final model, because its pipeline was built from 9 years of data (2007–2015) instead of just one year (2016) — making it more reliable for scoring future loans, even with a slightly lower test score. Full reasoning is in the Final Model Selection section.

In [26]:
#print(f"Exp2 addr_state dummies: {len([c for c in X_train_e2.columns if c.startswith('addr_state_')])}")

Exp2 addr_state dummies: 49


In [27]:
import joblib
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
MODEL_DIR = os.path.join(project_root, 'src', 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(xgb_e2, os.path.join(MODEL_DIR, 'xgb_modern.pkl'))
joblib.dump(list(X_train_e2.columns), os.path.join(MODEL_DIR, 'xgb_modern_features.pkl'))

print(" model saved to:", MODEL_DIR)
print(os.listdir(MODEL_DIR))

 model saved to: C:\Users\bhagyashree.s\Desktop\Machine_Learning_Projects\LendingClub_End_to_End_ML\src\models
['.ipynb_checkpoints', 'lgbm_with_tradeline.pkl', 'lgbm_with_tradeline_features.pkl', 'lr2_features.pkl', 'lr2_final.pkl', 'xgb_modern.pkl', 'xgb_modern_features.pkl', 'xgb_no_tradeline.pkl', 'xgb_no_tradeline_features.pkl', 'xgb_with_tradeline.pkl', 'xgb_with_tradeline_features.pkl']


In [28]:
print(f"Exp2 addr_state dummies: {len([c for c in X_train_e2.columns if c.startswith('addr_state_')])}")
print(f"Exp2 purpose dummies: {len([c for c in X_train_e2.columns if c.startswith('purpose_')])}")
print(f"Exp2 home_ownership dummies: {len([c for c in X_train_e2.columns if c.startswith('home_ownership_')])}")

Exp2 addr_state dummies: 49
Exp2 purpose dummies: 12
Exp2 home_ownership dummies: 3


In [29]:
# =========================================================
# SAVE EXPERIMENT 2 SPLITs
# =========================================================

import os

PARQUET_DIR = os.path.join(project_root, 'src', 'data', 'processed', 'parquet_chunks')
os.makedirs(PARQUET_DIR, exist_ok=True)

X_train_e2.to_parquet(os.path.join(PARQUET_DIR, 'X_train_e2.parquet'), index=False)
X_val_e2.to_parquet(os.path.join(PARQUET_DIR, 'X_val_e2.parquet'), index=False)
X_test_e2.to_parquet(os.path.join(PARQUET_DIR, 'X_test_e2.parquet'), index=False)

y_train_e2.to_frame(name='target').to_parquet(os.path.join(PARQUET_DIR, 'y_train_e2.parquet'), index=False)
y_val_e2.to_frame(name='target').to_parquet(os.path.join(PARQUET_DIR, 'y_val_e2.parquet'), index=False)
y_test_e2.to_frame(name='target').to_parquet(os.path.join(PARQUET_DIR, 'y_test_e2.parquet'), index=False)

print("Saved Experiment 2 splits to:", PARQUET_DIR)
print([f for f in os.listdir(PARQUET_DIR) if '_e2' in f])

Saved Experiment 2 splits to: C:\Users\bhagyashree.s\Desktop\Machine_Learning_Projects\LendingClub_End_to_End_ML\src\data\processed\parquet_chunks
['X_test_e2.parquet', 'X_train_e2.parquet', 'X_val_e2.parquet', 'y_test_e2.parquet', 'y_train_e2.parquet', 'y_val_e2.parquet']


### Why Feature Counts Differ Between Experiment 1 and Experiment 2

Experiment 1 and Experiment 2 train on different time windows (2007–2015 vs. 2016 only), so the final number of one-hot encoded columns is not expected to match exactly. This is expected behavior, not a data or pipeline error.

**How one-hot encoding works**

For any categorical column, the number of dummy columns created equals the number of distinct categories present in that specific training data, minus one (the first category is dropped, since it's implied when all other dummies are zero).

This means the dummy count for a column can only change if the actual set of categories present in the data changes.

| Column | Exp 1 (2007–2015) | Exp 2 (2016 only) | Difference | Reason |
|---|---|---|---|---|
| `addr_state` | 47 | 49 | +2 | Fewer states fell below the 500-row rare-category threshold in 2016 (mature, high-volume year) than across 2007–2015 (early years had low, uneven volume) |
| `purpose` | 13 | 12 | −1 | A loan purpose category (e.g. `educational`) was discontinued before 2016, so it doesn't appear in Experiment 2's training data |
| `home_ownership` | 3 | 3 | 0 | Categories are fixed and not time-sensitive — all values are equally present in both training windows |
| **Net total** | | | **1** | Matches the overall shape difference (144 vs. 143) |

**Takeaway**

The number of one-hot encoded columns depends on which categories actually appear in a given training slice, not on a fixed schema. Columns tied to time-sensitive factors (geography, product offerings) can differ in category count across training windows, while columns unrelated to time (like ownership status) stay consistent. The small difference in total feature count between Experiment 1 and Experiment 2 is fully explained by this, and does not indicate an error in either pipeline.